# 12 · Polytope: six-hour Climate-DT precipitation

**Objective:** Request a Northern Europe precipitation-rate subset from a specified climate simulation and integrate verified time intervals.

**Route:** Live / upgraded Climate-DT access / separate Polytope environment · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** native model-point and display-interpolated precipitation maps after rate metadata are confirmed. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/polytope.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Authenticate and select the Polytope environment

Use the separate `.venv-polytope` environment in the [setup guide](../docs/setup/README.md). It isolates upstream Zarr 2-dependent tooling from EDH's Zarr 3 environment. A DestinE login alone may not provide Climate-DT access; request the required upgrade.

The service provider's `destinepyauth` library supports `get_token('polytope')` and writes the expected `~/.polytopeapirc` token configuration. The following cell prompts for credentials, supports the provider's configured flow, and deliberately does not print the returned tokens. Re-authenticate when tokens expire. The official Polytope examples also supply a reviewed external authentication script; no missing local script or stdout parsing is used here.


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
from destinepyauth import get_token
_authentication = get_token("polytope")
del _authentication
print("Polytope authentication configuration prepared outside this repository.")


## 3. Build a specific simulation request

This preserves the original ScenarioMIP / SSP3-7.0 / IFS-NEMO / generation 1 example. Availability of these archive keys must be checked against the live catalogue. A January 2020 date in a scenario simulation is a simulated climate realization; it is not a reconstruction of the observed weather on that day. `standard` is an archive resolution label, not a promise of field-scale resolution.


In [ ]:
request = {
    "activity": "ScenarioMIP", "class": "d1", "dataset": "climate-dt",
    "date": "20200102", "experiment": "SSP3-7.0", "expver": "0001",
    "generation": "1", "levtype": "sfc", "model": "IFS-NEMO",
    "param": "260048", "realization": "1", "resolution": "standard",
    "stream": "clte", "time": "0100/0200/0300/0400/0500/0600", "type": "fc",
    "feature": {"type": "polygon", "axes": ["longitude", "latitude"], "shape": [[0,54],[35,54],[35,72],[0,72],[0,54]]},
}
print(request)


In [ ]:
import earthkit.data
address = os.getenv("POLYTOPE_ADDRESS", "polytope.lumi.apps.dte.destination-earth.eu")
if "/" in address or "@" in address:
    raise ValueError("Use the provider's documented Polytope hostname, without URL credentials.")
data = earthkit.data.from_source("polytope", "destination-earth", request, address=address, stream=False)
ds = data.to_xarray()
print({"variables": list(ds.data_vars), "sizes": dict(ds.sizes)})


## 4. Inspect rate units and the interval dimension

For a documented mean rate in `kg m−2 s−1`, multiplying by seconds gives millimetres of water equivalent. Only set `POLYTOPE_INTERVAL_SECONDS=3600` when the product describes each selected rate as the mean over a one-hour interval. If the values are instantaneous rates, integrating them needs a stated numerical rule and sufficient time coverage; this tutorial stops rather than assuming that six timestamps are six hourly totals.


In [ ]:
from examples.climate import precipitation_mm
rate = ds["tprate"]
print(rate.attrs)
time_dim = required_env("POLYTOPE_RATE_TIME_DIM")
if time_dim not in rate.dims or rate.sizes[time_dim] != 6:
    raise ValueError("Expected six rate intervals; inspect the returned schema before aggregation.")
units = str(rate.attrs.get("units", "")).replace(" ", "").lower()
if units not in {"kgm-2s-1", "kgm**-2s**-1", "kg/m^2/s", "kgm^-2s^-1"}:
    raise ValueError("Expected documented kg m-2 s-1. Verify metadata or adapt the conversion explicitly.")
seconds = float(required_env("POLYTOPE_INTERVAL_SECONDS"))
if seconds != 3600:
    raise ValueError("This six-hour teaching example expects six confirmed one-hour intervals.")
interval_mm = rate.copy(data=precipitation_mm(rate.values, "rate_kg_m2_s", seconds=seconds))
# skipna=False keeps incomplete points missing rather than treating absent rain as zero.
precip_6h = interval_mm.sum(time_dim, skipna=False).squeeze()
precip_6h.attrs["units"] = "mm / six verified hourly intervals"
print(precip_6h)


## 5. Visualize native points; interpolate only for display

A rectangular plotting grid is an interpolation artifact. It cannot increase the model's physical resolution. Retain the native values for analysis; HEALPix weighting depends on the actual returned grid and footprint.


In [ ]:
from scipy.interpolate import griddata
longitude, latitude, values = np.broadcast_arrays(precip_6h["longitude"].values, precip_6h["latitude"].values, precip_6h.values)
longitude, latitude, values = longitude.ravel(), latitude.ravel(), values.ravel()
valid = np.isfinite(longitude) & np.isfinite(latitude) & np.isfinite(values)
if valid.sum() < 4:
    raise ValueError("Too few valid model points for a display interpolation.")
grid_lon, grid_lat = np.meshgrid(np.linspace(0,35,120), np.linspace(54,72,80))
display_grid = griddata((longitude[valid], latitude[valid]), values[valid], (grid_lon, grid_lat), method="linear")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
native = axes[0].scatter(longitude[valid], latitude[valid], c=values[valid], s=5)
axes[0].set_title("Native model points")
fig.colorbar(native, ax=axes[0], label="Six-hour precipitation (mm)")
interpolated = axes[1].pcolormesh(grid_lon, grid_lat, display_grid, shading="auto")
axes[1].set_title("Interpolated display; no finer information")
fig.colorbar(interpolated, ax=axes[1], label="mm")
plt.show()
record_provenance("polytope-climate-dt.json", request=request, service_host=address, rate_metadata=rate.attrs, integration_seconds=seconds, time_dimension=time_dim, simulation="not an observation", display_interpolation="linear; native values retained", native_valid_points=int(valid.sum()))


## Interpret and verify

Parameter availability, conversion schema, time dimension and units require live verification. The archive request is retained as a documented starting point; update keys using official examples if the provider changes them. No uncertainty estimate follows from a single scenario/model/member. No regional mean is claimed without grid-area weighting.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [Official Polytope examples](https://github.com/destination-earth-digital-twins/polytope-examples)
- [Provider authentication library](https://github.com/SercoSPA/DestinE-Platform-AuthN)
- [Earthkit Polytope source](https://earthkit-data.readthedocs.io/en/stable/autoapi/earthkit/data/sources/polytope/index.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
